# Chapter 09 답안 양식. 회귀 분석으로 숫자 예측하기

> 이 내용을 `chapter09.ipynb`의 Markdown 셀로 작성합니다.

## 제출 정보
- 이름: 이상재
- GitHub ID: sangjae-lee97
- 작성일: 2026.09.22
- 최종 Notebook URL: https://github.com/sangjae-lee97/kant-axagent-study/blob/main/llm-data-analysis-course/chapter09/chapter09.ipynb

## 1. 예측 문제 정의
- Target: order_total
- 예측 단위: 주문 1건
- 예측 시점: 주문의 기본 정보와 고객 정보가 확보된 시점에서 최종 주문 금액을 예측한다.
- 예측 시점에 알 수 있는 정보:
order_month
order_dayofweek
age
payment_method
gender
city
- 예측 시점에 알 수 없는 정보:
order_total
line_total
quantity
unit_price
item_count
total_quantity
avg_unit_price
order_status
order_id
customer_id
product_id

### 나의 해석과 판단
왜 이 예측 시점을 선택했는지 작성하세요.

주문 기본 정보가 확보된 시점을 바탕으로 최종 주문 금액이 확보되기 전에 " 이 주문은 대략 얼마짜리가 될까?"를 예측하기 위해서

## 2. Feature Leakage Audit

| feature         | 예측 시점 사용 가능 | Target 계산 재료 | 사후 정보/ID | 최종 사용 | 판단 이유                                                   |
| --------------- | ----------- | ------------ | -------- | ----- | ------------------------------------------------------- |
| order_month     | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| order_dayofweek | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| age             | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| payment_method  | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| gender          | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| city            | O           | X            | X        | O     | 교육용 예측 시점에 사용 가능하다고 가정                                  |
| avg_unit_price  | X           | O            | O        | X     | 주문 상세에서 계산한 값으로 Target을 간접적으로 알려주는 목표 대리 변수             |
| customer_id     | O           | X            | O        | X     | 예측 시점에 존재할 수 있지만 고객을 구분하는 식별자이므로 제외                     |
| item_count      | X           | X            | O        | X     | 주문 상세가 확인된 뒤 계산되는 사후 집계값                                |
| line_total      | X           | O            | O        | X     | `order_total`을 구성하는 주문 상세 금액이므로 데이터 누수 발생               |
| order_id        | O           | X            | O        | X     | 주문을 구분하기 위한 식별자이며 예측에 의미 있는 일반 특성이 아님                   |
| order_status    | X           | X            | O        | X     | 주문 진행 이후 확정될 수 있는 사후 정보                                 |
| order_total     | X           | X            | O        | X     | 예측하려는 Target 자체이므로 feature로 사용할 수 없음                    |
| product_id      | X           | X            | O        | X     | 주문 상세가 확인되어야 알 수 있는 상품 식별자                              |
| quantity        | X           | O            | O        | X     | `line_total = quantity × unit_price`에 사용되는 Target 계산 재료 |
| total_quantity  | X           | O            | O        | X     | 주문 상세의 quantity를 집계한 값으로 Target을 간접적으로 알려주는 목표 대리 변수    |
| unit_price      | X           | O            | O        | X     | `line_total = quantity × unit_price`에 사용되는 Target 계산 재료 |


- 금지 feature overlap 개수: 0개
- 가장 위험하다고 판단한 feature: order_total
- 이유: 우리가 알아야할 최종 정보이므로 이걸 사용하는 순간 머신러닝이 아닌 이미 아는 정보를 바탕으로 끼워맞추기가 됨

![Leakage 점검](images/step02_leakage.png)

## 3. Target 생성과 관계 검증

### 계산 기준

```text
line_total = quantity * unit_price
order_total = order_id별 line_total 합계
```

In [12]:
import pandas as pd
import numpy as np

# 데이터 불러오기
order_items = pd.read_csv("../data/processed/order_items_clean.csv")

# 계산값 만들기
expected_line_total = order_items["quantity"] * order_items["unit_price"]

# 기존 line_total과 계산값 비교
is_match = np.isclose(
    order_items["line_total"],
    expected_line_total
)

print("전체 행 수:", len(order_items))
print("일치 행 수:", is_match.sum())
print("불일치 행 수:", (~is_match).sum())

if is_match.all():
    print("line_total 검증 결과: 일치")
else:
    print("line_total 검증 결과: 불일치")

전체 행 수: 764
일치 행 수: 764
불일치 행 수: 0
line_total 검증 결과: 일치


- line_total 검증 결과: 일치
- orders ↔ target 관계 검증 결과: one_to_one
- orders → customers 관계 검증 결과: many_to_one

### 나의 해석과 판단
관계 검증 없이 병합만 수행하면 어떤 문제가 생길 수 있는지 작성하세요.

관계 검증 없이 병합하면 중복 행 증가나 행 누락이 발생할 수 있고, 그 결과 주문 금액과 같은 집계값이 실제보다 크게 또는 작게 계산될 수 있다. 따라서 병합 전에 one_to_one, many_to_one 등의 관계가 예상과 일치하는지 확인해야 한다.


## 4. Train / Final Test 분할

- Train 시작일:2025-07-09
- Train 종료일:2026-04-13
- Train 행 수:244
- Final Test 시작일:2026-04-14
- Final Test 종료일:2026-07-08
- Final Test 행 수:56
- 같은 달력 날짜 중복 여부: 없음

In [13]:
import pandas as pd

# 주문 데이터 불러오기
orders = pd.read_csv("../data/processed/orders_clean.csv")

# 날짜형 변환
orders["order_date"] = pd.to_datetime(orders["order_date"])

# 날짜 기준으로 Train / Test 분리
train = orders[
    (orders["order_date"] >= "2025-07-09") &
    (orders["order_date"] <= "2026-04-13")
].copy()

test = orders[
    (orders["order_date"] >= "2026-04-14") &
    (orders["order_date"] <= "2026-07-08")
].copy()

# 같은 날짜가 양쪽에 존재하는지 확인
overlap_dates = set(train["order_date"].dt.date) & set(test["order_date"].dt.date)

print("Train 행 수:", len(train))
print("Test 행 수:", len(test))
print("같은 달력 날짜 중복 개수:", len(overlap_dates))
print("중복 날짜:", sorted(overlap_dates))

print("Train 마지막 날짜:", train["order_date"].max())
print("Test 첫 날짜:", test["order_date"].min())

print(
    "시간 순서 검증:",
    "PASS" if train["order_date"].max() < test["order_date"].min() else "FAIL"
)

Train 행 수: 244
Test 행 수: 56
같은 달력 날짜 중복 개수: 0
중복 날짜: []
Train 마지막 날짜: 2026-04-13 00:00:00
Test 첫 날짜: 2026-04-14 00:00:00
시간 순서 검증: PASS


### 분할 판단
왜 이번 문제에서 시간 순서 분할을 사용하는지 작성하세요.

우리의 목표는 예측이기 때문에 먼 과거의 데이터를 바탕으로 현재를 예측하는게 목표와 부합하기 때문

![시간 분할](images/step04_split.png)

## 5. Train TimeSeriesSplit 후보 비교

In [14]:
from src.regression import (
    load_regression_source_data,
    build_regression_dataset,
    split_model_data_by_time,
    make_regression_models,
    cross_validate_regression_models,
    select_diagnostic_model,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
)

# 실제 processed 폴더 경로
processed_dir = r"C:\dev\kant-axagent-study\llm-data-analysis-course\data\processed"

# 데이터 불러오기
data = load_regression_source_data(processed_dir)

# 모델링 데이터 생성
model_data = build_regression_dataset(
    customers=data["customers"],
    orders=data["orders"],
    order_items=data["order_items"],
)

# Train / Test 시간 분할
train_data, test_data = split_model_data_by_time(
    model_data,
    test_size=0.2
)

# Train 데이터
X_train = train_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()

# 모델 생성
models = make_regression_models(random_state=42)

# TimeSeriesSplit CV
cv_summary = cross_validate_regression_models(
    models,
    X_train,
    y_train
)

# 비베이스라인 모델 선택
selected_model = select_diagnostic_model(cv_summary)

print(cv_summary)
print()
print("Selected Model:", selected_model)

               model  n_splits    cv_MAE_mean    cv_MAE_std  cv_R2_mean
0      Baseline Mean         5  423004.808743  44153.043466   -0.022821
1      Random Forest         5  437352.803929  26318.793429   -0.109687
2  Linear Regression         5  461386.286905  77792.799324   -0.310517

Selected Model: Random Forest


| 모델                |  CV MAE 평균 | CV MAE 표준편차 | CV R² 평균 | 비고        |
| ----------------- | ---------: | ----------: | -------: | --------- |
| Baseline Mean     | 423,004.81 |   44,153.04 |  -0.0228 | 비교 기준     |
| Linear Regression | 461,386.29 |   77,792.80 |  -0.3105 | 비베이스라인 후보 |
| Random Forest     | 437,352.80 |   26,318.79 |  -0.1097 | 선택 모델     |

### Train CV로 선택한 비베이스라인 모델
- Selected Model: Random Forest

- 선택 기준:
비베이스라인 후보인 Linear Regression과 Random Forest 중
CV MAE 평균이 더 낮은 모델을 선택하였다.

- Final Test 결과를 보기 전에 선택했는가: 예

### 나의 해석과 판단
CV 평균뿐 아니라 표준편차도 함께 봐야 하는 이유를 작성하세요.

CV 평균은 모델의 전체적인 성능 수준을 보여주고, 표준편차는 각 검증 구간에서 성능이 얼마나 안정적으로 유지되는지 보여줍니다.

평균이 좋아도 표준편차가 크면 시기에 따라 성능 차이가 많이 날 수 있으므로, 모델의 안정성을 판단하기 위해 둘을 함께 봐야 합니다.

## 6. Final Test: Baseline vs Frozen Model

In [15]:
from src.regression import train_and_evaluate_models

# Final Test 데이터 준비
X_test = test_data[FEATURE_COLUMNS].copy()
y_test = test_data[TARGET_COLUMN].copy()

# Baseline Mean과 이미 선택한 Frozen Model만 최종 평가
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)

# 결과 확인
display(model_comparison)

,model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
0,Baseline Mean,baseline,416266.191884,443449.648712,522189.699620,-0.001121,0.00
1,Random Forest,selected_by_train_cv,326438.478132,458819.917234,548190.446723,-0.103298,-3.47




| 모델 | selection role | Train MAE | Test MAE | Test RMSE | Test R² | Baseline 대비 MAE 개선율 |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| Baseline Mean | baseline | | | | | |
| Frozen Selected Model | selected_by_train_cv | | | | | |

![최종 회귀 평가](./images/step06_final_metrics.png)

### 결과 관찰

- Baseline Mean의 Test MAE는 약 443,450원이다.
- Random Forest의 Test MAE는 약 458,820원이다.
- Random Forest의 Test RMSE는 약 548,190원이다.
- Random Forest의 Test R²는 -0.1033이다.
- Baseline 대비 MAE 개선율은 -3.47%이다.
- Random Forest의 Train MAE는 약 326,438원이지만 Test MAE는 약 458,820원으로 증가했다.

### 나의 해석과 판단

Random Forest는 Train CV에서 선택된 모델이지만, Final Test에서는 Baseline Mean보다 좋은 성능을 보이지 못했다.

MAE 기준으로 Baseline보다 약 3.47% 성능이 악화되었으며, R²도 음수로 나타나 평균값을 예측하는 단순한 방법보다 Final Test 데이터를 더 잘 설명하지 못했다.

또한 Train MAE보다 Test MAE가 크게 증가했기 때문에 과적합 가능성도 추가로 확인할 필요가 있다.

현재 결과만 보면 Random Forest가 Baseline보다 추가적인 예측 가치를 제공한다고 보기 어렵다.

### 업무·분석적 의미

현재 Random Forest의 Test MAE는 약 458,820원으로, 실제 주문 금액과 예측 금액 사이에 평균적으로 약 46만 원 정도의 차이가 발생한다. 또한 RMSE가 약 548,190원으로 MAE보다 높아 일부 주문에서는 더 큰 예측 오차가 발생할 가능성이 있다.

특히 Baseline Mean보다 MAE가 3.47% 높고 R²도 음수이므로, 현재 모델은 단순히 평균 주문 금액을 예측하는 방법보다 추가적인 예측 가치를 보여주지 못했다. 따라서 현재 수준의 모델을 개별 주문 금액을 정확히 예측해야 하는 업무에 바로 활용하기는 어렵다.

실제 업무에 적용하려면 예측 시점에 사용할 수 있으면서 주문 금액과 관련된 추가 변수를 확보하고, 새로운 미래 데이터에서 다시 성능을 검증할 필요가 있다.

## 7. 대표 오차 사례

In [16]:
from src.regression import create_prediction_result

# 6번에서 평가한 Random Forest의 예측값 가져오기
selected_predictions = predictions[selected_model]

# 실제값, 예측값, residual, absolute error 계산
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=selected_predictions,
    model_name=selected_model,
)

# 절대오차가 가장 큰 2건 확인
top_errors = (
    prediction_result
    .sort_values("abs_error", ascending=False)
    .head(2)
)

display(
    top_errors[
        [
            "order_date",
            "actual_order_total",
            "predicted_order_total",
            "residual",
            "abs_error",
        ]
    ]
)

,order_date,actual_order_total,predicted_order_total,residual,abs_error
0,2026-06-15,1877000,605528.715709,1.271471e+06,1.271471e+06
1,2026-06-19,1767000,756889.654706,1.010110e+06,1.010110e+06




### 사례 1

- 실제값: 1,877,000원
- 예측값: 약 605,529원
- residual: 약 +1,271,471원
- absolute error: 약 1,271,471원
- 관찰: 실제 주문금액이 예측값보다 크게 높아 약 127만 원을 과소예측했다.
- 가능한 이유 후보: 현재 사용 중인 주문 월, 요일, 나이, 결제수단, 성별, 지역만으로는 이러한 고액 주문의 특성을 충분히 설명하지 못했을 가능성이 있다.
- 추가 확인 사항: 해당 주문이 일반 주문보다 금액이 큰 이유를 설명할 수 있는 예측 시점 이전의 추가 정보가 존재하는지 확인할 필요가 있다.

### 사례 2

- 실제값: 1,767,000원
- 예측값: 약 756,890원
- residual: 약 +1,010,110원
- absolute error: 약 1,010,110원
- 관찰: 실제 주문금액이 예측값보다 높아 약 101만 원을 과소예측했다.
- 가능한 이유 후보: 사례 1과 마찬가지로 현재 feature만으로 고액 주문을 구분할 수 있는 충분한 신호가 부족했을 가능성이 있다.
- 추가 확인 사항: 고액 주문에서 비슷한 과소예측이 반복되는지, 그리고 이를 설명할 수 있는 추가 feature가 있는지 확인할 필요가 있다.

> 원인을 데이터가 직접 보여 주지 않는다면 사실처럼 단정하지 않습니다.

## 8. Validation Evidence

`reports/ch09_regression_validation.csv` 기준으로 작성하세요.

| check | 결과 | 의미 |
|---|---:|---|
| `forbidden_feature_overlap` | 0 / PASS | 금지 feature가 모델 입력에 없음 |
| `strict_train_before_test` | True / PASS | Train이 Final Test보다 시간상 앞에 있음 |
| `selected_model_exists_in_train_cv` | True / PASS | Random Forest가 실제 Train CV 후보였음 |
| `final_test_contains_baseline` | True / PASS | Final Test에 Baseline Mean이 포함됨 |
| `final_test_contains_frozen_selected_model` | True / PASS | CV에서 고정한 Random Forest만 최종 모델로 평가됨 |
| `test_rows_for_r2` | 56 / PASS | R²를 계산하기에 충분한 Test 데이터가 있음 |

### FAIL 항목
- 없음 / 있음: 없음
- 있다면 원인과 수정 내용: 없음

## 9. Evidence와 공개 범위

### 생성된 Evidence
- [x] `ch09_regression_split_summary.csv`
- [x] `ch09_regression_feature_audit.csv`
- [x] `ch09_regression_cv_summary.csv`
- [x] `ch09_regression_model_comparison.csv`
- [x] `ch09_regression_validation.csv`
- [x] `ch09_regression_checklist.csv`

### Internal 파일
- [x] `ch09_regression_model_data_internal.csv`
- [x] `ch09_regression_predictions_internal.csv`

### 공개 결과
- [x] `ch09_regression_report.md`
- [x] `ch09_actual_vs_predicted.png`
- [x] `ch09_residual_histogram.png`

### 공개 범위 판단
왜 `order_id`가 포함된 예측 결과를 Internal로 구분해야 하는지 작성하세요.
order_id는 개별 주문을 식별하기 위한 내부 식별자이므로, 공개 결과에 반드시 필요한 정보는 아닙니다. 또한 예측 결과와 함께 공개될 경우 특정 주문을 추적할 수 있는 가능성이 있어 불필요한 식별 정보 노출 위험이 생길 수 있기때문.


## 10. 최종 사용 판단

- [ ] 현재 수준에서 참고용 사용 가능
- [ ] 추가 검증 후 사용 가능
- [x] 현재 데이터로는 사용 보류

### 판단 근거
1. Random Forest의 Test MAE는 약 458,820원으로, Baseline Mean의 약 443,450원보다 높아 Baseline 대비 MAE가 3.47% 악화되었다.
2. Random Forest의 Test R²가 -0.1033으로 나타나 단순 평균 예측보다 Final Test 데이터를 더 잘 설명하지 못했다.
3. Validation 항목은 모두 PASS하여 평가 절차 자체는 적절했지만, 대표 오차 사례에서 약 101만~127만 원의 큰 과소예측이 확인되었다.

### 업무적 위험
현재 모델을 실제 주문 금액 예측에 사용할 경우 평균적으로 약 46만 원의 오차가 발생하며, 일부 주문에서는 100만 원 이상의 큰 오차가 발생할 수 있다. 따라서 예산 산정이나 매출 예측처럼 주문 금액의 정확성이 중요한 업무에 사용하면 잘못된 판단으로 이어질 위험이 있다.

### 현재 데이터의 한계
현재 feature는 주문 월, 요일, 고객 나이, 결제수단, 성별, 지역으로 제한되어 있다. 이러한 정보만으로는 주문 금액, 특히 고액 주문의 차이를 충분히 설명하기 어려울 가능성이 있다.

### 다음 개선 우선순위
1. 예측 시점 이전에 사용할 수 있으면서 주문 금액과 관련된 추가 feature를 탐색한다.
2. 고액 주문에서 큰 과소예측이 반복되는지 잔차와 오차 패턴을 추가 분석한다.
3. 개선 후에는 현재 Final Test를 반복적인 모델 선택에 사용하지 않고 새로운 미래 기간의 데이터로 다시 검증한다.

## 11. 전체 재실행 확인

실행 명령:

```powershell
python scripts/run_regression_analysis.py
```

- 실행 성공 여부: 성공
- Train CV Selected Model: Random Forest
- Final Validation 전체 PASS 여부: 6개 전부 PASS
- Notebook 결과와 스크립트 결과의 일관성: 일치

### 결과 확인

전체 회귀 분석 스크립트를 처음부터 다시 실행한 결과,
Train CV에서 선택된 모델은 Random Forest로 Notebook 결과와 동일했다.

Final Test에서도 Random Forest의 Test MAE는 약 458,820원,
Test RMSE는 약 548,190원, Test R²는 -0.1033,
Baseline 대비 MAE 개선율은 -3.47%로 Notebook에서 확인한 결과와 일치했다.

또한 자동 Validation의 6개 항목이 모두 PASS하였으며,
예측 오차 결과와 Evidence 파일 및 진단 그래프도 정상적으로 생성되었다.

따라서 현재 Chapter 09 분석 결과는 전체 재실행을 통해 재현됨을 확인했다.

---

## 최종 체크

- [x] Target과 예측 시점을 정의했습니다.
- [x] Leakage Audit을 수행했습니다.
- [x] Target 계산과 관계 검증을 확인했습니다.
- [x] Train과 Final Test가 시간 순서로 엄격히 분리되었습니다.
- [x] 전처리가 Pipeline 내부에서 학습됩니다.
- [x] Train TimeSeriesSplit으로 후보를 비교했습니다.
- [x] Final Test 전에 Selected Model을 고정했습니다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했습니다.
- [x] MAE, RMSE, R²를 함께 해석했습니다.
- [x] 대표 오차 사례를 관찰과 가설로 구분했습니다.
- [x] Validation Evidence가 모두 PASS입니다.
- [x] Internal 결과와 공개 결과를 구분했습니다.
- [x] 낮은 성능도 숨기지 않았습니다.
- [x] 전체 스크립트를 재실행했습니다.
- [x] 최종 Notebook URL을 제출합니다.